In [2]:
import json
from datasets import load_dataset

with open("/workspace/data/distilled/pmc_patients/pmc_patients_distilled_prepared.jsonl") as f:
    raw = json.loads(f.readline())

dataset = load_dataset(
    "loknezmonzter/pmc-patients-distilled-medgemma-22B",
    split="train",
    revision="distilled-medgemma-15k"
)

In [18]:
sample = dataset.filter(lambda x: x['id'] == "2527548-1")
sample[0]['id']

Filter:   0%|          | 0/13377 [00:00<?, ? examples/s]

'2527548-1'

In [19]:
hf_sample = sample[0]

In [21]:
print("ORIGINAL data keys:", list(raw.keys()))
print("HF data keys:", list(hf_sample.keys()))

ORIGINAL data keys: ['id', 'status', 'data', 'error', 'text']
HF data keys: ['id', 'status', 'data', 'error', 'text']


In [22]:
print("ORIGINAL text[:200]:", raw.get("text", "")[:200])
print("HF text[:200]:", hf_sample.get("text", "")[:200])

ORIGINAL text[:200]: During routine dissection of the right upper limb of a 69 year old male cadaver, we observed an accessory muscle belly took origin from the radial side of the FDS tendon to the index finger, at the le
HF text[:200]: During routine dissection of the right upper limb of a 69 year old male cadaver, we observed an accessory muscle belly took origin from the radial side of the FDS tendon to the index finger, at the le


In [23]:
print("MATCH:", raw.get("text") == hf_sample.get("text"))

MATCH: True


In [3]:
import json

# Exact JSON schema - convert to JSON string
JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}
SCHEMA_STRING = json.dumps(JSON_SCHEMA, indent=4)

# Finalized system prompt
SYSTEM_PROMPT = (
    "You are an expert clinical informatician. "
    f"Extract data strictly into this JSON schema:\n\n{SCHEMA_STRING}\n\n"
    "CRITICAL RULES:\n"
    "1. Use ONLY double quotes for all JSON keys and string values.\n"
    "2. Response MUST start with '{' and end with '}'.\n"
    "3. Output raw JSON only — no markdown, no code blocks.\n"
    "4. Provide values for ALL keys in the schema.\n"
    "5. Apostrophes in clinical terms (e.g., patient's) are allowed inside double-quoted strings.\n"
    "6. Extract at max 10 most clinically significant relationships only. "
    "Prioritize: diagnosis > treatment > symptoms > history.\n"
)

In [ ]:
import torch
import json
from transformers import (
    Mistral3ForConditionalGeneration,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from dotenv import load_dotenv
from datasets import load_dataset

load_dotenv()

model_id = "mistralai/Ministral-3-3B-Instruct-2512-BF16"

# Same BnB config that WORKS for you
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Same loading path that WORKS for you
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Prepare for k-bit training (required before adding LoRA)
model = prepare_model_for_kbit_training(model)

def format_example(example):
    """
    Format columns into standard Hugging Face conversation dictionaries.
    The output dict MUST expose a single structural column names "messages"
    """
    # Ground truth from medgemma extractions
    target_json = json.dumps(example['data'], indent=2)

     # Build prompt string with chat template (includes assistant header)
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CONTEXT:\n{example['text']}"}
    ]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True  # Adds the assistant role header (e.g., <|assistant|>)
    )

    # Completion is just the JSON output + EOS
    completion_text = target_json + tokenizer.eos_token

    return {
        "prompt": prompt_text,      
        "completion": completion_text  
    }

# Load the dataset
dataset = load_dataset(
    "loknezmonzter/pmc-patients-distilled-medgemma-22B",
    split="train",
    revision="distilled-medgemma-15k"
)

# Randomize the train dataset 
# Split entire train set into smaller train and test
train = dataset.shuffle(seed=42) 
split = train.train_test_split(test_size=0.1, seed=42)

# Use 2500 random records for training and 250 random records for evaluation
train_split = split["train"].select(range(2500))
eval_split = split["test"].select(range(25))
eval_split = eval_split.shuffle(seed=3407)

train = train_split.map(format_example, desc="Formatting train")
eval_set = eval_split.map(format_example, desc="Formatting eval")

# Take ONE example and test
example = eval_set[3]
text = example["prompt"] + example["completion"]

inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=4096).to("cuda")

model.eval()
with torch.no_grad():
    outputs = model(**inputs, labels=inputs["input_ids"])
    print(f"Manual loss: {outputs.loss.item()}")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Formatting eval:   0%|          | 0/25 [00:00<?, ? examples/s]

Manual loss: 1.3524048328399658


: 

In [1]:
!nvidia-smi

Mon May 25 07:13:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.12              Driver Version: 550.90.12      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:00:07.0 Off |                    0 |
| 30%   31C    P8             25W /  300W |       2MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----